# YouTube Spam Detection Notebook (Refactored)

Notebook propre et reproductible pour classifier des commentaires YouTube en `Ham` ou `Spam`.


## 1. Imports et configuration


In [ ]:
import glob
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from sklearn.pipeline import Pipeline

RANDOM_STATE = 365
TEST_SIZE = 0.2
DATASET_GLOB = 'youtube-dataset/*.csv'

sns.set_theme(style='whitegrid')


## 2. Chargement et préparation des données


In [ ]:
def load_dataset(pattern: str = DATASET_GLOB) -> pd.DataFrame:
    files = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(f'Aucun fichier trouvé avec le pattern: {pattern}')

    frames = []
    for f in files:
        df = pd.read_csv(f)
        frames.append(df[['CONTENT', 'CLASS']].dropna())

    data = pd.concat(frames, ignore_index=True)
    data['CONTENT'] = data['CONTENT'].astype(str)
    data['CLASS'] = data['CLASS'].astype(int)
    return data


data = load_dataset()
print(f'Nombre total de lignes: {len(data)}')
data.head()


In [ ]:
data['CLASS'].value_counts().rename({0: 'Ham', 1: 'Spam'})


## 3. Split train/test


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data['CONTENT'],
    data['CLASS'],
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=data['CLASS'],
)

print('Train shape:', X_train.shape)
print('Test shape :', X_test.shape)
print('
Distribution y_train:')
print(y_train.value_counts(normalize=True).round(3))
print('
Distribution y_test:')
print(y_test.value_counts(normalize=True).round(3))


## 4. Pipelines modèles

On compare deux variantes Naive Bayes:
- `MultinomialNB`
- `ComplementNB` (souvent plus robuste sur texte déséquilibré)


In [ ]:
def make_pipeline(model):
    return Pipeline([
        ('tfidf', TfidfVectorizer(
            lowercase=True,
            strip_accents='unicode',
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.98,
        )),
        ('clf', model),
    ])

pipelines = {
    'MultinomialNB': make_pipeline(MultinomialNB(alpha=1.0)),
    'ComplementNB': make_pipeline(ComplementNB(alpha=0.5)),
}


## 5. Validation croisée (sur train)


In [ ]:
cv_results = {}

for name, pipe in pipelines.items():
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='f1_macro')
    cv_results[name] = {
        'mean_f1_macro': scores.mean(),
        'std_f1_macro': scores.std(),
    }

cv_df = pd.DataFrame(cv_results).T.sort_values('mean_f1_macro', ascending=False)
cv_df


## 6. Entraînement final et évaluation test


In [ ]:
best_model_name = cv_df.index[0]
best_model = pipelines[best_model_name]

best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

print(f'Modèle retenu: {best_model_name}')
print('
Classification report:')
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam'], digits=4))


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=['Ham', 'Spam'],
    cmap='Blues',
    ax=ax,
)
ax.set_title(f'Confusion Matrix - {best_model_name}')
plt.show()


## 7. Prédictions sur nouveaux commentaires


In [ ]:
examples = [
    'This song is amazing, I love it!',
    'Win money now!!! Click here: http://spam-link.com',
    'Great tutorial, thanks for sharing.',
    'Subscribe to my channel and get free gifts!'
]

preds = best_model.predict(examples)

pd.DataFrame({
    'comment': examples,
    'prediction_label': ['Spam' if p == 1 else 'Ham' for p in preds],
    'prediction_class': preds,
})


## 8. Sauvegarde du modèle


In [ ]:
artifacts_dir = Path('artifacts')
artifacts_dir.mkdir(exist_ok=True)

model_path = artifacts_dir / 'spam_model_from_notebook.joblib'
joblib.dump(best_model, model_path)

print(f'Modèle sauvegardé: {model_path}')
